# Kirundi news topic classification: Transformer fine-tuning (GPU)
Runs on **Kaggle** (Settings > Accelerator: GPU T4 x2 or P100, Internet: On) or Colab.
Steps: 1) setup, 2) smoke test, 3) experiments (4 configs x 3 seeds), 4) aggregate, 5) download results.

**Before running:** upload the 6 files from your local `data/processed/` folder
(`raw_train.csv`, `raw_val.csv`, `raw_test.csv`, `raw_official_train.csv`, `raw_official_test.csv`, `raw_report.json`)
as a **private** Kaggle dataset named `kirnews-processed` (article copyright belongs to the news sources, so keep it private).

In [ ]:
import os, subprocess, torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
REPO = "https://github.com/Delucie-Rurangwa/kirundi-news-classification.git"
if not os.path.exists("kirundi-news-classification"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("kirundi-news-classification")
# Kaggle: /kaggle/input/kirnews-processed   |   Colab: upload files to /content/processed and change this
DATA_DIR = "/kaggle/input/kirnews-processed"
print(os.listdir(DATA_DIR))

In [ ]:
!pip -q install transformers accelerate tabulate

## 1. Smoke test (about 1-2 minutes). If this fails, fix it before the long runs.

In [ ]:
!python src/finetune_transformer.py --model Davlan/afro-xlmr-base --data_dir {DATA_DIR} --out_dir results/smoke --smoke --name smoke_s42

## 2. Experiments
| ID | Model | Max length | Loss | Question it answers |
|---|---|---|---|---|
| E1 | `xlm-roberta-base` | 256 | plain CE | How does a general multilingual model do? |
| E2 | `Davlan/afro-xlmr-base` | 256 | plain CE | Does African-language adaptation help? |
| E3 | `Davlan/afro-xlmr-base` | 256 | class-weighted CE | Does weighting help the rare classes? |
| E4 | `Davlan/afro-xlmr-base` | 512 | class-weighted CE | Does reading more of the article help? |

3 seeds each (42, 43, 44) because the test set is tiny and one run is noisy.

In [ ]:
import subprocess
SEEDS = [42, 43, 44]
EXPS = {
 "E1_xlmr_base_len256":            dict(model="xlm-roberta-base",       max_len=256, weighted=False, batch=16, accum=1),
 "E2_afroxlmr_base_len256":        dict(model="Davlan/afro-xlmr-base",  max_len=256, weighted=False, batch=16, accum=1),
 "E3_afroxlmr_base_len256_wloss":  dict(model="Davlan/afro-xlmr-base",  max_len=256, weighted=True,  batch=16, accum=1),
 "E4_afroxlmr_base_len512_wloss":  dict(model="Davlan/afro-xlmr-base",  max_len=512, weighted=True,  batch=8,  accum=2),
}
for name, c in EXPS.items():
    for seed in SEEDS:
        run = f"{name}_s{seed}"
        if os.path.exists(f"results/transformers/result_{run}.json"):
            print("skip (done):", run); continue
        cmd = ["python", "src/finetune_transformer.py", "--model", c["model"], "--data_dir", DATA_DIR,
               "--max_len", str(c["max_len"]), "--batch", str(c["batch"]), "--grad_accum", str(c["accum"]),
               "--seed", str(seed), "--name", run]
        if c["weighted"]: cmd.append("--weighted_loss")
        print("RUNNING", run)
        subprocess.run(cmd, check=True)

## 3. Aggregate the results into the experiments table

In [ ]:
!python src/aggregate_results.py

## 4. Save the best model and package the results

In [ ]:
# Save the final model (E2 or E3, whichever you choose after seeing the table) for the web app.
# Edit --model/--weighted_loss to match the winner, then upload /kaggle/working/final_model to the Hugging Face Hub.
!python src/finetune_transformer.py --model Davlan/afro-xlmr-base --data_dir {DATA_DIR} --max_len 256 --weighted_loss --seed 42 --name final_s42 --out_dir results/final --save_dir /kaggle/working/final_model
!zip -rq /kaggle/working/results_transformers.zip results